<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-03-prompting/lesson-3.3-model-routing/notebooks/GCP_Capstone_3.3_Model_Routing.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.3 Chain-of-Thought & Model Routing
**Netsetos GenAI Engineering — GCP Capstone**

Thinking mode, complexity classifier, model router, 84% cost savings.


## Setup


In [ ]:
!pip install -q google-genai pydantic scipy
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal
import json, time

client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')


## Cell 1: Thinking Mode — Inspect Thought Tokens


In [ ]:
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='What is 17 * 23 + 45 * 12?',
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_budget=2048, include_thoughts=True)),
)

for part in response.candidates[0].content.parts:
    if not part.text: continue
    label = 'THOUGHT' if part.thought else 'ANSWER'
    print(f'{label}: {part.text[:100]}')

meta = response.usage_metadata
print(f'\nInput: {meta.prompt_token_count} | Output: {meta.candidates_token_count} | Thinking: {meta.thoughts_token_count}')


## Cell 2: Thinking Budget Comparison


In [ ]:
question = 'What is 17 * 23 + 456 - 89?'

for budget in [0, 1024, 4096, 8192]:
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=question,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=budget)))
    think = r.usage_metadata.thoughts_token_count or 0
    out = r.usage_metadata.candidates_token_count or 0
    print(f'  budget={budget:<6} think={think:<5} out={out:<5} answer={(r.text or "").strip()[:40]}')


## Cell 3: Structured CoT — Reasoning Before Answer


In [ ]:
class ReasonedAnswer(BaseModel):
    reasoning: str = Field(description='Step-by-step reasoning')
    answer: str = Field(description='Final answer')
    confidence: Literal['high','medium','low']

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='A train travels 120km in 1.5 hours. Average speed?',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=ReasonedAnswer,
        temperature=0.1,
        thinking_config=types.ThinkingConfig(thinking_budget=0)))

result = r.parsed
print(f'Reasoning: {result.reasoning}')
print(f'Answer: {result.answer}')
print(f'Confidence: {result.confidence}')


## Cell 4: Complexity Classifier


In [ ]:
def classify_complexity(query):
    r = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=f"""Classify complexity:
SIMPLE: Factual lookups, greetings, definitions, classification
MEDIUM: Explanations, comparisons, summaries, standard code
COMPLEX: Multi-step math, architecture, debugging, proofs

Query: {query}""",
        config=types.GenerateContentConfig(
            response_mime_type='text/x.enum',
            response_schema={'type':'STRING','enum':['SIMPLE','MEDIUM','COMPLEX']},
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    return r.text

test_queries = [
    'What is Python?',
    'Compare REST vs GraphQL for microservices',
    'Prove sqrt(2) is irrational',
    'What time is it?',
    'Design a distributed cache with consistency guarantees',
]
for q in test_queries:
    print(f'  {classify_complexity(q):<8} | {q}')


## Cell 5: Model Router


In [ ]:
ROUTING_TABLE = {
    'SIMPLE':  {'model':'gemini-3.1-flash-lite','budget':0,   'temp':0.1,'max':512},
    'MEDIUM':  {'model':'gemini-3.6-flash',     'budget':1024,'temp':0.3,'max':2048},
    'COMPLEX': {'model':'gemini-3.1-pro-preview',       'budget':8192,'temp':0.7,'max':16384},
}

def route_and_generate(query, system_instruction=''):
    complexity = classify_complexity(query)
    cfg = ROUTING_TABLE[complexity]
    r = client.models.generate_content(
        model=cfg['model'], contents=query,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=cfg['temp'], max_output_tokens=cfg['max'],
            thinking_config=types.ThinkingConfig(thinking_budget=cfg['budget'])))
    meta = r.usage_metadata
    return {'text':r.text, 'complexity':complexity, 'model':cfg['model'],
            'think_tokens':meta.thoughts_token_count or 0, 'out_tokens':meta.candidates_token_count}

for q in test_queries[:3]:
    result = route_and_generate(q)
    print(f'{result["complexity"]:<8} -> {result["model"]:<25} think={result["think_tokens"]}')
    print(f'  {result["text"][:80]}...\n')


## Cell 6: Cost Calculator


In [ ]:
PRICING = {
    'gemini-3.1-flash-lite': {'input':0.25, 'output':1.50},
    'gemini-3.6-flash':      {'input':1.50, 'output':7.50},
    'gemini-3.1-pro-preview':        {'input':2.00, 'output':12.00},
}

def cost_per_1k(dist):
    total = 0
    for tier, d in dist.items():
        n = d['queries']
        p = PRICING[d['model']]
        c = n*d['avg_in']/1e6*p['input'] + n*(d['avg_out']+d['avg_think'])/1e6*p['output']
        print(f'  {tier:<10} {n:>4}q | {d["model"]:<25} | ${c:.3f}')
        total += c
    print(f'  TOTAL: ${total:.3f}')
    return total

print('ROUTED (70/25/5):')
routed = cost_per_1k({
    'simple':  {'queries':700,'model':'gemini-3.1-flash-lite','avg_in':200,'avg_out':100,'avg_think':0},
    'medium':  {'queries':250,'model':'gemini-3.6-flash','avg_in':400,'avg_out':250,'avg_think':1024},
    'complex': {'queries':50, 'model':'gemini-3.1-pro-preview','avg_in':600,'avg_out':500,'avg_think':8192},
})
print('\nALWAYS-PRO:')
pro = cost_per_1k({
    'all': {'queries':1000,'model':'gemini-3.1-pro-preview','avg_in':300,'avg_out':200,'avg_think':4000},
})
print(f'\nSAVINGS: {(1-routed/pro)*100:.1f}% vs always-Pro')


## Cell 7: Cascade Router (Advanced)


In [ ]:
from pydantic import BaseModel

# Cascade router: try the cheapest model first, escalate only when it isn't confident.
# NOTE: token logprobs (the ideal confidence signal) are NOT exposed for Gemini 3.x on
# Vertex -- response_logprobs raises "Logprobs is not supported for this model"; they
# exist only on older 2.0-era models. So we cascade on the model's SELF-REPORTED
# confidence via structured output (a practical proxy -- calibrate thresholds on real data).

class ConfidentAnswer(BaseModel):
    answer: str
    confidence: float  # 0.0-1.0: how sure the answer is correct AND complete

def cascade_route(query):
    stages = [
        ('gemini-3.1-flash-lite', 0),      # cheapest, no thinking
        ('gemini-3.6-flash', 2048),        # mid tier, some thinking
        ('gemini-3.1-pro-preview', 8192),  # last resort, deep thinking
    ]
    thresholds = [0.75, 0.60]  # escalate if self-confidence is below this
    for i, (model, budget) in enumerate(stages):
        r = client.models.generate_content(
            model=model,
            contents='Answer the query, then set confidence (0-1) to how sure you '
                     'are your answer is correct and complete. Query: ' + query,
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_budget=budget),
                response_mime_type='application/json',
                response_schema=ConfidentAnswer))
        a = r.parsed
        if i == len(stages) - 1:
            return a.answer, model, 'last_resort'
        if a.confidence >= thresholds[i]:
            return a.answer, model, 'confident (%.2f)' % a.confidence
        print('  Escalating from %s (confidence=%.2f)' % (model, a.confidence))
    return a.answer, model, 'escalated'

for q in ['What is Python?', 'Prove sqrt(2) is irrational']:
    text, model, reason = cascade_route(q)
    print('%s (%s): %s...' % (model, reason, text[:60]))

## Cell 8: Complete model_router.py Module


In [ ]:
# Production Module
def classify(query):
    r = client.models.generate_content(
        model='gemini-3.1-flash-lite', contents=f'Classify: {query}',
        config=types.GenerateContentConfig(
            response_mime_type='text/x.enum',
            response_schema={'type':'STRING','enum':['SIMPLE','MEDIUM','COMPLEX']},
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    return r.text

def route(query, system_instruction='', schema=None):
    complexity = classify(query)
    cfg = ROUTING_TABLE[complexity]
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=cfg['temp'], max_output_tokens=cfg['max'],
        thinking_config=types.ThinkingConfig(thinking_budget=cfg['budget']))
    if schema:
        config.response_mime_type = 'application/json'
        config.response_schema = schema
    r = client.models.generate_content(model=cfg['model'], contents=query, config=config)
    return {'text':r.text, 'complexity':complexity, 'model':cfg['model'],
            'think_tokens':r.usage_metadata.thoughts_token_count or 0}

print('Module ready: classify(), route()')
print('Integration: route(query, system_instruction=PERSONA, schema=RAGAnswer)')


## ✅ Module 3 Complete!

- ✅ 3.1: system_instruction, temperature, ThinkingConfig, personas, A/B testing
- ✅ 3.2: JSON schema, Pydantic, enum, few-shot, anyOf, $ref
- ✅ 3.3: Thinking mode, structured CoT, complexity classifier, model router, 84% savings

**Three production modules:**
- `prompt_config.py` — 4 preset configs + persona router
- `structured_output.py` — RAGAnswer, DocMetadata, config factories
- `model_router.py` — classify(), route(), cost tracking

**Next: Module 4 — Building RAG Three Ways**
